# Dicts, sets, tuples: data structure patterns

**M0 · Toolbox & Submission Craft** — [AI Olympiad Curriculum](https://ai-olympiad-curriculum.netlify.app)

~45 min

## The four containers, honestly

`list` — ordered, mutable, allows duplicates: the default sequence. `tuple` — the same but immutable: safe dict keys, multi-value returns. `dict` — key → value lookup: index anything by anything. `set` — unique items with instant membership tests: dedupe and "have I seen this?".

Rule of thumb: **list** for data you walk through in order, **dict** for lookup by name, **set** for membership and dedupe, **tuple** when the contents must not change — or must serve as a dict key.



In [ ]:
label2id = {"cat": 0, "dog": 1}      # dict
label2id["bird"] = 2                # add a key
label2id.get("fox", -1)             # -1 — .get returns a default instead of KeyError
"cat" in label2id                   # True — "in" on a dict checks KEYS
pair = (0.91, "cat")                # tuple — e.g. a (score, label) returned together

## Dict patterns: counting and grouping

Two dict idioms appear in every dataset pipeline. **Counting**: `d[k] = d.get(k, 0) + 1` — `.get(key, default)` avoids the KeyError on first sighting; called with no default, `d.get(k)` returns `None`. **Grouping**: collect items into per-key lists with `setdefault` — `d.setdefault(k, v)` returns `d[k]`, inserting `v` first if `k` is missing.



In [ ]:
labels = ["cat", "dog", "cat", "bird", "dog", "cat"]

counts = {}
for lab in labels:
    counts[lab] = counts.get(lab, 0) + 1
counts                          # {'cat': 3, 'dog': 2, 'bird': 1}

by_first = {}                   # group words by first letter
for w in ["cat", "dog", "crow"]:
    by_first.setdefault(w[0], []).append(w)
by_first                        # {'c': ['cat', 'crow'], 'd': ['dog']}

`collections.Counter` does the counting for you (`Counter(labels).most_common(3)`) and `defaultdict(list)` removes the setdefault — both appear in the next lesson. Write the plain version by hand first so you know what they hide.

Looping over a dict yields its **keys**; `.items()` yields (key, value) pairs; `.values()` yields values. Invert a dict — swap keys and values — with a comprehension: `{v: k for k, v in d.items()}`.



## Tuples and unpacking

Unpacking splits a sequence into names — in assignments, in `for` targets, in `return`. It is why `a, b = b, a` swaps with no temporary variable: the right side is fully built before assignment happens.



In [ ]:
def top_two(scores):
    return max(scores), min(scores)      # returns ONE tuple

hi, lo = top_two([0.4, 0.9, 0.7])        # unpacked at the call site
a, b = 1, 2
a, b = b, a                              # swap

for i, (name, sc) in enumerate([("cat", 0.9), ("dog", 0.7)]):
    print(i, name, sc)                   # nested unpacking in the for-target

Use `_` for values you don't need (`name, _ = pair`), and `first, *rest = xs` to grab the head plus everything else — `*rest` collects the remainder into a list.



## Sets: membership and dedupe

`set` stores each item once and answers `x in s` in ~O(1) — a hash lookup, ~instant regardless of the set's size — where a list scans all n items. Deduplicating 10⁶ items against a set is instant; against a list it is roughly 10¹² comparisons: seconds versus hours.



In [ ]:
seen = {"cat", "dog"}
"cat" in seen                        # True — constant time
seen.add("bird"); seen.add("cat")    # 'cat' was already there — sets keep one copy

a, b = {1, 2, 3}, {2, 3, 4}
a | b                                # {1, 2, 3, 4}  union
a & b                                # {2, 3}        intersection
a - b                                # {1}           difference
list({3, 1, 2, 1})                   # dedupe — but order is not guaranteed

- Set items and dict keys must be hashable: numbers, strings, tuples work — lists and dicts don't.
- `{1}` is a set but `{}` is an empty **dict** — write `set()` for an empty set.
- Sets are unordered: `sorted(s)` when order matters, never rely on iteration order.



## Sorting with keys

`sorted(xs, key=f)` orders by `f(x)` — the value each item maps to. A lambda picks out a field; `reverse=True` flips the order. A tuple key sorts lexicographically: first element decides, ties fall to the second.



In [ ]:
pairs = [("cat", 0.9), ("dog", 0.7), ("bird", 0.9)]
sorted(pairs, key=lambda p: p[1])            # by score, ascending
sorted(pairs, key=lambda p: -p[1])           # descending, via negation
sorted(pairs, key=lambda p: (-p[1], p[0]))   # score desc, name asc on ties

counts = {"cat": 3, "dog": 2}
sorted(counts, key=counts.get)               # ['dog', 'cat'] — dict keys by value

> **Note.** `sorted` returns a new list and leaves the original alone; `xs.sort()` mutates `xs` and returns `None`. Both accept `key=` and `reverse=`.



## Aliasing: the bug that bites everyone

`b = a` never copies a container — both names point to **the same object**, so `b.append(3)` shows up in `a` too. Copy with `a.copy()` (one level) or `copy.deepcopy(a)` (nested containers).



In [ ]:
a = [1, 2]
b = a          # b is the SAME list
b.append(3)
a              # [1, 2, 3] — surprise

c = a.copy()   # a real copy
c.append(4)    # a is untouched

d = [[0] * 3] * 2        # looks like two rows...
d[0][1] = 9              # ...but d[1][1] is also 9 — both rows are the same list
d = [[0] * 3 for _ in range(2)]   # fix: build each row separately

- `def f(xs=[])` shares one default list across all calls — mutate it once and it leaks into every later call. Default to `xs=None`.
- Slices copy shallowly: `a[:]` copies a flat list, but a list of lists still shares the inner lists — `copy.deepcopy` fixes that.
- Functions receive the same object too: a helper that mutates its argument changes the caller's data. Copy at the boundary, or document the mutation.



### ✏️ What does `{k: v for k, v in zip(['a', 'b'], [1, 2])}` produce?

**A.** An error
**B.** {'a': 1, 'b': 2}
**C.** [('a', 1), ('b', 2)]
**D.** {'a', 'b', 1, 2}

<details><summary>Answer</summary>

**B.** {'a': 1, 'b': 2}

zip pairs the items; the dict comprehension turns each pair into key: value.

</details>

### ✏️ d = {'a': 1}. What does d.get('b') return?

**A.** 'b'
**B.** 0
**C.** None
**D.** KeyError

<details><summary>Answer</summary>

**C.** None

.get() returns None (or a default you pass) instead of raising; d['b'] would raise KeyError.

</details>

### ✏️ `a = [1, 2]; b = a; b.append(3)`. What is `a` now?

**A.** [1, 2]
**B.** An error
**C.** [3]
**D.** [1, 2, 3]

<details><summary>Answer</summary>

**D.** [1, 2, 3]

b and a refer to the same list object, so changing b changes a.

</details>

---

**Go deeper**
- [The Python Tutorial (official)](https://docs.python.org/3/tutorial/)